SEMI-SUPERVISED ANCHOR ONLY

In [ ]:
"""Anchor-only semi-supervised PG19 frequency inference.

This script implements the black-box six-class experiment under two strict
constraints:

1. The learning algorithm receives labels only for ``Very high``, ``Very low``,
   and ``Term not present``. Every ``High``, ``Moderate``, and ``Low`` training
   label is masked before the estimator is fitted.
2. The only timing measurement used to construct predictors is
   ``rag_end_to_end_latency_ms``. Internal pipeline latencies, query text,
   semantic category, answer properties, and corpus-frequency fields are never
   predictors.

Because a conventional self-training classifier cannot predict classes that
never occur in its labeled seed set, the model is an ordinal semi-supervised
estimator. A Random Forest regressor learns a latent ordinal score from the
three labeled anchors. Five-fold target-noun-grouped cross-fitting produces
training scores, and the unlabeled score distribution determines five
equal-prior ordinal cut points. The intermediate labels High, Moderate, and
Low are therefore induced from the ordered unlabeled distribution rather than
supplied to the learner.

The source experiment is balanced by construction, so the six equal class
priors are fixed before fitting. Ground-truth labels for masked classes are
used only for the stratified experimental split and final held-out evaluation.
"""

from __future__ import annotations

import argparse
import json
import re
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold


plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

RANDOM_STATE = 42
QUERY_COLUMN = "query_id"
TARGET_COLUMN = "frequency"
END_TO_END_COLUMN = "rag_end_to_end_latency_ms"
EXPECTED_K_VALUES = (5, 20, 50, 100)
EXPECTED_ITERATIONS = (1, 2)

FREQUENCY_ORDER = (
    "Very high",
    "High",
    "Moderate",
    "Low",
    "Very low",
    "Term not present",
)

LABELED_ANCHORS = (
    "Very high",
    "Very low",
    "Term not present",
)

MASKED_TRAINING_CLASSES = (
    "High",
    "Moderate",
    "Low",
)

ORDINAL_VALUE = {
    class_name: float(index)
    for index, class_name in enumerate(FREQUENCY_ORDER)
}


def infer_target_noun_group(query_id: str) -> str:
    """Recover the repeated target-noun group encoded by the query identifier.

    Identifiers use blocks of ten templates per target noun and reset across
    query-length conditions. Length is removed so every query containing the
    same target noun remains in one train/test group.
    """

    match = re.fullmatch(r"(.+)__(short|medium|long)__(.+)__(\d+)", query_id)
    if match is None:
        raise ValueError(f"Unexpected query_id format: {query_id!r}")

    frequency_key, _, category_key, numeric_suffix = match.groups()
    target_index = int(numeric_suffix) // 10
    return f"{frequency_key}__{category_key}__target_{target_index}"


def build_end_to_end_query_table(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Create one query vector from its eight end-to-end measurements only."""

    required_columns = {
        QUERY_COLUMN,
        TARGET_COLUMN,
        "k",
        "iteration",
        END_TO_END_COLUMN,
    }
    missing = sorted(required_columns.difference(raw_df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if raw_df[END_TO_END_COLUMN].isna().any():
        raise ValueError(f"{END_TO_END_COLUMN} contains missing values")
    if (raw_df[END_TO_END_COLUMN] < 0).any():
        raise ValueError(f"{END_TO_END_COLUMN} contains negative values")

    observed_k = tuple(sorted(raw_df["k"].unique().tolist()))
    observed_iterations = tuple(sorted(raw_df["iteration"].unique().tolist()))
    if observed_k != EXPECTED_K_VALUES:
        raise ValueError(f"Expected k={EXPECTED_K_VALUES}; observed {observed_k}")
    if observed_iterations != EXPECTED_ITERATIONS:
        raise ValueError(
            f"Expected iterations={EXPECTED_ITERATIONS}; observed {observed_iterations}"
        )

    raw_df = raw_df.copy()
    raw_df["target_noun_group"] = raw_df[QUERY_COLUMN].map(infer_target_noun_group)

    for column in (TARGET_COLUMN, "target_noun_group"):
        maximum_unique = raw_df.groupby(QUERY_COLUMN)[column].nunique().max()
        if maximum_unique != 1:
            raise ValueError(f"{column!r} is not constant within query_id")

    repeated_counts = raw_df.groupby(QUERY_COLUMN).size()
    expected_repetitions = len(EXPECTED_K_VALUES) * len(EXPECTED_ITERATIONS)
    if not repeated_counts.eq(expected_repetitions).all():
        raise ValueError(
            f"Every query must have {expected_repetitions} measurements; "
            f"observed range {repeated_counts.min()}--{repeated_counts.max()}"
        )

    identifiers = raw_df.groupby(QUERY_COLUMN).agg(
        frequency=(TARGET_COLUMN, "first"),
        target_noun_group=("target_noun_group", "first"),
    )

    pivot = raw_df.pivot(
        index=QUERY_COLUMN,
        columns=["k", "iteration"],
        values=END_TO_END_COLUMN,
    )
    expected_pairs = [
        (k_value, iteration)
        for k_value in EXPECTED_K_VALUES
        for iteration in EXPECTED_ITERATIONS
    ]
    pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(expected_pairs))
    pivot.columns = [
        f"e2e_k{k_value}_iteration{iteration}_ms"
        for k_value, iteration in pivot.columns
    ]
    raw_feature_columns = pivot.columns.tolist()

    query_df = identifiers.join(pivot).reset_index()

    # This monotone transform stabilizes the long right tail while preserving
    # the end-to-end timing information used by tree split thresholds.
    model_feature_columns: list[str] = []
    for raw_column in raw_feature_columns:
        transformed_column = f"log1p_{raw_column}"
        query_df[transformed_column] = np.log1p(query_df[raw_column])
        model_feature_columns.append(transformed_column)

    group_audit = query_df.groupby("target_noun_group").agg(
        number_of_queries=(QUERY_COLUMN, "size"),
        number_of_labels=(TARGET_COLUMN, "nunique"),
    )
    if not group_audit["number_of_queries"].eq(30).all():
        raise ValueError("Expected 30 queries for every target-noun group")
    if not group_audit["number_of_labels"].eq(1).all():
        raise ValueError("Each target-noun group must have exactly one frequency label")

    return query_df, model_feature_columns


def target_noun_disjoint_split(
    query_df: pd.DataFrame,
    test_fraction: float = 0.20,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create a class-balanced split with no target noun shared across sets."""

    rng = np.random.default_rng(RANDOM_STATE)
    test_groups: list[str] = []

    for class_name in FREQUENCY_ORDER:
        class_groups = np.asarray(
            sorted(
                query_df.loc[
                    query_df[TARGET_COLUMN] == class_name,
                    "target_noun_group",
                ].unique()
            )
        )
        rng.shuffle(class_groups)
        number_test_groups = max(1, int(round(len(class_groups) * test_fraction)))
        test_groups.extend(class_groups[:number_test_groups].tolist())

    is_test = query_df["target_noun_group"].isin(test_groups)
    train_df = query_df.loc[~is_test].copy().reset_index(drop=True)
    test_df = query_df.loc[is_test].copy().reset_index(drop=True)

    overlap = set(train_df["target_noun_group"]).intersection(
        test_df["target_noun_group"]
    )
    if overlap:
        raise AssertionError(f"Target-noun leakage detected: {sorted(overlap)}")

    split_audit = (
        pd.concat(
            [
                train_df.assign(split="train"),
                test_df.assign(split="test"),
            ],
            ignore_index=True,
        )
        .groupby(["split", TARGET_COLUMN], observed=False)
        .agg(
            queries=(QUERY_COLUMN, "size"),
            target_nouns=("target_noun_group", "nunique"),
        )
        .reindex(
            pd.MultiIndex.from_product(
                [["train", "test"], FREQUENCY_ORDER],
                names=["split", TARGET_COLUMN],
            )
        )
        .reset_index()
    )
    return train_df, test_df, split_audit


def make_seed_labels(train_df: pd.DataFrame) -> pd.Series:
    """Expose only the three permitted anchor labels to the estimator."""

    seed_labels = train_df[TARGET_COLUMN].where(
        train_df[TARGET_COLUMN].isin(LABELED_ANCHORS),
        other=pd.NA,
    )

    observed_seed_classes = set(seed_labels.dropna().unique())
    if observed_seed_classes != set(LABELED_ANCHORS):
        raise AssertionError(
            f"Expected only {LABELED_ANCHORS}; observed {sorted(observed_seed_classes)}"
        )
    if seed_labels[train_df[TARGET_COLUMN].isin(MASKED_TRAINING_CLASSES)].notna().any():
        raise AssertionError("A High, Moderate, or Low training label was exposed")

    return seed_labels


def make_anchor_regressor() -> RandomForestRegressor:
    """Use the supplied Random Forest architecture for the latent ordinal score."""

    return RandomForestRegressor(
        n_estimators=200,
        min_samples_split=15,
        min_samples_leaf=5,
        max_features="sqrt",
        max_depth=30,
        criterion="squared_error",
        bootstrap=True,
        random_state=RANDOM_STATE,
        # A single worker avoids platform-dependent floating-point reduction
        # order and makes the emitted cut points byte-for-byte reproducible.
        n_jobs=1,
    )


@dataclass
class AnchorOnlyOrdinalModel:
    """Cross-fitted ordinal learner with unlabeled quantile boundaries."""

    n_splits: int = 5
    class_priors: tuple[float, ...] = (1 / 6,) * 6

    def fit(
        self,
        X: pd.DataFrame,
        seed_labels: pd.Series,
        target_noun_groups: pd.Series,
    ) -> "AnchorOnlyOrdinalModel":
        if len(X) != len(seed_labels) or len(X) != len(target_noun_groups):
            raise ValueError("X, seed_labels, and target_noun_groups must align")
        if not np.isclose(sum(self.class_priors), 1.0):
            raise ValueError("class_priors must sum to one")

        seed_labels = seed_labels.reset_index(drop=True)
        X = X.reset_index(drop=True)
        target_noun_groups = target_noun_groups.reset_index(drop=True)

        labeled_mask = seed_labels.notna().to_numpy()
        labeled_indices = np.flatnonzero(labeled_mask)
        unlabeled_indices = np.flatnonzero(~labeled_mask)
        labeled_names = seed_labels.iloc[labeled_indices].astype(str).to_numpy()

        splitter = StratifiedGroupKFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        cross_fitted_scores = np.full(len(X), np.nan, dtype=float)
        unlabeled_score_sum = np.zeros(len(unlabeled_indices), dtype=float)
        self.estimators_: list[RandomForestRegressor] = []

        for fit_positions, validation_positions in splitter.split(
            X.iloc[labeled_indices],
            labeled_names,
            groups=target_noun_groups.iloc[labeled_indices],
        ):
            fit_indices = labeled_indices[fit_positions]
            validation_indices = labeled_indices[validation_positions]

            estimator = make_anchor_regressor()
            estimator.fit(
                X.iloc[fit_indices],
                np.asarray(
                    [
                        ORDINAL_VALUE[class_name]
                        for class_name in seed_labels.iloc[fit_indices].astype(str)
                    ]
                ),
            )
            cross_fitted_scores[validation_indices] = estimator.predict(
                X.iloc[validation_indices]
            )
            unlabeled_score_sum += estimator.predict(X.iloc[unlabeled_indices])
            self.estimators_.append(estimator)

        cross_fitted_scores[unlabeled_indices] = (
            unlabeled_score_sum / len(self.estimators_)
        )
        if np.isnan(cross_fitted_scores).any():
            raise AssertionError("Cross-fitted training scores contain missing values")

        cumulative_priors = np.cumsum(self.class_priors)[:-1]
        self.cut_points_ = np.quantile(cross_fitted_scores, cumulative_priors)
        if not np.all(np.diff(self.cut_points_) > 0):
            raise ValueError("Ordinal cut points are not strictly increasing")

        self.training_scores_ = cross_fitted_scores
        self.initial_labeled_ = int(labeled_mask.sum())
        self.initial_unlabeled_ = int((~labeled_mask).sum())
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def predict_scores(self, X: pd.DataFrame) -> np.ndarray:
        X = X.loc[:, self.feature_names_in_]
        return np.mean(
            [estimator.predict(X) for estimator in self.estimators_],
            axis=0,
        )

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        scores = self.predict_scores(X)
        class_indices = np.digitize(scores, self.cut_points_)
        return np.asarray(FREQUENCY_ORDER, dtype=object)[class_indices]


def ordinal_score(true_class: str, predicted_class: str) -> float:
    positions = {label: index for index, label in enumerate(FREQUENCY_ORDER)}
    distance = abs(positions[true_class] - positions[predicted_class])
    if distance == 0:
        return 1.0
    if distance == 1:
        return 0.8
    if distance == 2:
        return 0.6
    return 0.0


def ordinally_weighted_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(
        np.mean(
            [
                ordinal_score(true_class, predicted_class)
                for true_class, predicted_class in zip(y_true, y_pred)
            ]
        )
    )


def ordinal_weighted_recall_per_class(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    results: dict[str, float] = {}
    for true_class in FREQUENCY_ORDER:
        predictions = y_pred[y_true == true_class]
        results[true_class] = float(
            np.mean(
                [ordinal_score(true_class, predicted) for predicted in predictions]
            )
        )
    return results


def ordinal_weighted_precision_per_class(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    results: dict[str, float] = {}
    for predicted_class in FREQUENCY_ORDER:
        true_classes = y_true[y_pred == predicted_class]
        results[predicted_class] = (
            float(
                np.mean(
                    [ordinal_score(true_class, predicted_class) for true_class in true_classes]
                )
            )
            if len(true_classes)
            else np.nan
        )
    return results


def harmonic_mean(precision: float, recall: float) -> float:
    if np.isnan(precision) or np.isnan(recall):
        return np.nan
    if precision + recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


def evaluate_and_save(
    model: AnchorOnlyOrdinalModel,
    test_df: pd.DataFrame,
    feature_columns: list[str],
    split_audit: pd.DataFrame,
    seed_audit: pd.DataFrame,
    output_directory: Path,
) -> dict:
    output_directory.mkdir(parents=True, exist_ok=True)

    y_true = test_df[TARGET_COLUMN].to_numpy()
    y_pred = model.predict(test_df[feature_columns])
    latent_scores = model.predict_scores(test_df[feature_columns])

    normal_report = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=list(FREQUENCY_ORDER),
            output_dict=True,
            zero_division=0,
        )
    ).T.loc[
        list(FREQUENCY_ORDER),
        ["precision", "recall", "f1-score", "support"],
    ]

    ow_precision = ordinal_weighted_precision_per_class(y_true, y_pred)
    ow_recall = ordinal_weighted_recall_per_class(y_true, y_pred)
    ow_f1 = {
        class_name: harmonic_mean(
            ow_precision[class_name],
            ow_recall[class_name],
        )
        for class_name in FREQUENCY_ORDER
    }
    ordinal_report = pd.DataFrame(
        {
            "OW Precision": [ow_precision[name] for name in FREQUENCY_ORDER],
            "OW Recall": [ow_recall[name] for name in FREQUENCY_ORDER],
            "OW F1": [ow_f1[name] for name in FREQUENCY_ORDER],
            "support": [normal_report.loc[name, "support"] for name in FREQUENCY_ORDER],
        },
        index=FREQUENCY_ORDER,
    )

    normal_metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(FREQUENCY_ORDER),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(FREQUENCY_ORDER),
                average="weighted",
                zero_division=0,
            )
        ),
    }
    ordinal_metrics = {
        "ow_accuracy": ordinally_weighted_accuracy(y_true, y_pred),
        "macro_ow_precision": float(np.nanmean(list(ow_precision.values()))),
        "macro_ow_recall": float(np.nanmean(list(ow_recall.values()))),
        "macro_ow_f1": float(np.nanmean(list(ow_f1.values()))),
    }

    matrix = confusion_matrix(y_true, y_pred, labels=list(FREQUENCY_ORDER))
    matrix_df = pd.DataFrame(
        matrix,
        index=pd.Index(FREQUENCY_ORDER, name="True frequency"),
        columns=pd.Index(FREQUENCY_ORDER, name="Predicted frequency"),
    )

    predictions = test_df[
        [QUERY_COLUMN, "target_noun_group", TARGET_COLUMN]
    ].copy()
    predictions["predicted_frequency"] = y_pred
    predictions["latent_ordinal_score"] = latent_scores

    normal_report.to_csv(output_directory / "normal_classification_report.csv")
    ordinal_report.to_csv(
        output_directory / "ordinal_weighted_classification_report.csv"
    )
    matrix_df.to_csv(output_directory / "confusion_matrix.csv")
    predictions.to_csv(output_directory / "held_out_predictions.csv", index=False)
    split_audit.to_csv(output_directory / "split_audit.csv", index=False)
    seed_audit.to_csv(output_directory / "training_label_audit.csv", index=False)

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=list(FREQUENCY_ORDER),
    ).plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=35,
        colorbar=False,
        values_format="d",
    )
    ax.set_title("PG19 smol cache black-box frequency inference", pad=12)
    ax.set_xlabel("Predicted frequency")
    ax.set_ylabel("True frequency")
    for text in ax.texts:
        text.set_fontsize(24)
    for label in ax.get_xticklabels():
        label.set_ha("right")
        label.set_rotation_mode("anchor")
    fig.tight_layout()
    fig.savefig(output_directory / "confusion_matrix.png", dpi=300)
    fig.savefig(output_directory / "confusion_matrix.pdf", bbox_inches="tight")
    plt.close(fig)

    result = {
        "method": "anchor-only ordinal semi-supervised quantile model",
        "random_state": RANDOM_STATE,
        "raw_timing_columns_used": [END_TO_END_COLUMN],
        "model_feature_columns": feature_columns,
        "labeled_anchor_classes": list(LABELED_ANCHORS),
        "masked_training_classes": list(MASKED_TRAINING_CLASSES),
        "number_initially_labeled": model.initial_labeled_,
        "number_initially_unlabeled": model.initial_unlabeled_,
        "ordinal_cut_points": model.cut_points_.tolist(),
        "normal_metrics": normal_metrics,
        "ordinal_weighted_metrics": ordinal_metrics,
    }
    with (output_directory / "metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)

    print("\nANCHOR-ONLY SEMI-SUPERVISED FIT")
    print("Raw timing columns used:", [END_TO_END_COLUMN])
    print("Labeled classes:", list(LABELED_ANCHORS))
    print("Masked classes:", list(MASKED_TRAINING_CLASSES))
    print("Initially labeled:", model.initial_labeled_)
    print("Initially unlabeled:", model.initial_unlabeled_)
    print("Ordinal cut points:", np.round(model.cut_points_, 6).tolist())

    print("\nNORMAL METRICS")
    for name, value in normal_metrics.items():
        print(f"{name}: {value:.6f}")

    print("\nORDINAL WEIGHTED METRICS")
    for name, value in ordinal_metrics.items():
        print(f"{name}: {value:.6f}")

    print("\nNORMAL CLASSIFICATION REPORT")
    print(normal_report.round(4).to_string())

    print("\nORDINAL WEIGHTED CLASSIFICATION REPORT")
    print(ordinal_report.round(4).to_string())
    print(f"\nResults written to: {output_directory.resolve()}")
    return result


def run_experiment(csv_path: Path, output_directory: Path) -> dict:
    raw_df = pd.read_csv(csv_path)
    query_df, feature_columns = build_end_to_end_query_table(raw_df)
    train_df, test_df, split_audit = target_noun_disjoint_split(query_df)

    seed_labels = make_seed_labels(train_df)
    seed_audit = (
        pd.DataFrame(
            {
                "true_frequency_for_audit_only": train_df[TARGET_COLUMN],
                "label_exposed_to_model": seed_labels.fillna("UNLABELED"),
            }
        )
        .groupby(
            ["true_frequency_for_audit_only", "label_exposed_to_model"],
            observed=False,
        )
        .size()
        .rename("queries")
        .reset_index()
    )

    model = AnchorOnlyOrdinalModel(n_splits=5)
    model.fit(
        train_df[feature_columns],
        seed_labels,
        train_df["target_noun_group"],
    )

    return evaluate_and_save(
        model,
        test_df,
        feature_columns,
        split_audit,
        seed_audit,
        output_directory,
    )


def parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--csv",
        type=Path,
        default=Path("per_query_rag_llm_latencies_cache_smol_pg19.csv"),
        help="Path to the per-execution latency CSV.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("pg19_smol_cache_black_box_freq_results"),
        help="Directory for reports, metrics, predictions, and figures.",
    )
    # parse_known_args (instead of parse_args) discards arguments it does not
    # recognize, such as the "--f=/path/to/kernel-....json" flag that Jupyter
    # injects into sys.argv when this script is run inside a notebook cell or
    # via %run. Without this, argparse raises SystemExit on that flag.
    arguments, _unrecognized = parser.parse_known_args()
    return arguments


if __name__ == "__main__":
    arguments = parse_arguments()
    run_experiment(arguments.csv, arguments.output)